In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

# Load GPT-2 tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
C:\Users\User\anaconda3\Lib\site-packages\peft\tuners\lora\layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [2]:
import pandas as pd
from datasets import Dataset

# Load the original CSVs
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("valid.csv")

# ✅ Limit to 1000 training samples and 200 validation samples
train_df = train_df.sample(n=1000, random_state=42).reset_index(drop=True)
valid_df = valid_df.sample(n=200, random_state=42).reset_index(drop=True)

# Format the data for GPT-style conversation
train_df["text"] = train_df.apply(lambda row: f"User: {row['query']}\nBot: {row['answers']}", axis=1)
valid_df["text"] = valid_df.apply(lambda row: f"User: {row['query']}\nBot: {row['answers']}", axis=1)

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[["text"]])
valid_dataset = Dataset.from_pandas(valid_df[["text"]])

In [3]:
# GPT-2 doesn't have a padding token — we set one manually
tokenizer.pad_token = tokenizer.eos_token

In [4]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    tokens["labels"] = tokens["input_ids"]  # Important for training loss
    return tokens

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [5]:
from transformers import Trainer, TrainingArguments

# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",                # Where to save the model
    overwrite_output_dir=True,             # Overwrite old results
    num_train_epochs=3,                    # Train for 3 passes over the data
    per_device_train_batch_size=4,         # You can change based on your RAM
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir="./logs",                  # Where to save logs
    evaluation_strategy="epoch",           # Run eval after each epoch
    save_strategy="epoch",                 # Save model after each epoch
    save_total_limit=2,                    # Keep only last 2 checkpoints
    fp16=False                             # Set True if using GPU (FP16 mode)
)

C:\Users\User\anaconda3\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
)

# Train!
trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,0.536781
2,1.709100,0.479098
3,1.709100,0.458987


TrainOutput(global_step=750, training_loss=1.308793914794922, metrics={'train_runtime': 8455.9991, 'train_samples_per_second': 0.355, 'train_steps_per_second': 0.089, 'total_flos': 393297002496000.0, 'train_loss': 1.308793914794922, 'epoch': 3.0})

In [7]:
import torch

# Take 10 samples from validation set
sample_inputs = valid_dataset.select(range(10))

generated_responses = []
actual_responses = []

for sample in sample_inputs:
    input_text = sample["text"]
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=256).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=100,
            num_return_sequences=1,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_responses.append(decoded_output.split("Bot:")[-1].strip())
    actual_responses.append(sample["text"].split("Bot:")[-1].strip())


In [8]:
!pip install evaluate bert_score

  Obtaining dependency information for bert_score from https://files.pythonhosted.org/packages/c6/8c/bc5457de4c004b1a623b31f7bc8d0375fb699b7d67df11879098b4b7b7c8/bert_score-0.3.13-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/61.1 kB ? eta -:--:--
   ------------- -------------------------- 20.5/61.1 kB ? eta -:--:--
   ---------------------------------------- 61.1/61.1 kB 1.1 MB/s eta 0:00:00


In [10]:
!pip install -U datasets transformers bert_score

  Obtaining dependency information for datasets from https://files.pythonhosted.org/packages/b4/83/50abe521eb75744a01efe2ebe836a4b61f4df37941a776f650f291aabdf9/datasets-3.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for pyarrow>=15.0.0 from https://files.pythonhosted.org/packages/ff/77/e62aebd343238863f2c9f080ad2ef6ace25c919c6ab383436b5b81cbeef7/pyarrow-19.0.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for requests>=2.32.2 from https://files.pythonhosted.org/packages/f9/9b/335f9764261e915ed497fcdeb11df5dfd6f7bf257d4a6a2a686d80da4d54/requests-2.32.3-py3-none-any.whl.metadata
  Obtaining dependency information for tqdm>=4.66.3 from https://files.pythonhosted.org/packages/d0/30/dc54f88dd4a2b5dc8a0279bdd7270e735851848b762aeb1c1184ed1f6b14/tqdm-4.67.1-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     --------------------- ------------------ 30.7/57.7 kB 1.3 MB/s eta 0:00:01
     -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
tensorflow-intel 2.18.0 requires keras>=3.5.0, which is not installed.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.
s3fs 2023.4.0 requires fsspec==2023.4.0, but you have fsspec 2024.12.0 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
voila 0.5.7 requires jupyter-server<3,>=2.0.0, but you have jupyter-server 1.23.4 which is incompatible.


In [12]:
!pip install rouge_score

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24972 sha256=505ca2e07072ca23da4f5c34fa0175db90001ad7e40ffd3cbe14da853593713f
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\1e\19\43\8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
from datasets import load_metric
import bert_score

# Load ROUGE
rouge = load_metric("rouge")

# Compute ROUGE-L
rouge_result = rouge.compute(predictions=generated_responses, references=actual_responses, use_stemmer=True)
print("ROUGE-L:", round(rouge_result["rougeL"].mid.fmeasure, 4))

# BERTScore
P, R, F1 = bert_score.score(generated_responses, actual_responses, lang="en")
print("BERTScore (F1):", round(F1.mean().item(), 4))

ROUGE-L: 0.8898


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\User\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]